# Pipeline MLCQ — Visão geral dos dados

Este notebook resume o fluxo: **CSV original** → **busca do código nos repositórios** → **CSV com snippets** → **geração de SQL para a base**.

## 1. CSV original

Carregamento e visão geral do dataset original (metadados: repo, commit, path, start_line, end_line — **sem trecho de código**).

In [29]:
from pathlib import Path
import pandas as pd

BASE = Path.cwd().resolve()
original = pd.read_csv(BASE / "mlcq_original_dataset.csv", sep=";")
original.head()

,id,reviewer_id,sample_id,smell,severity,review_timestamp,type,code_name,repository,commit_hash,path,start_line,end_line,link,is_from_industry_relevant_project
0,526,6,5771277,feature envy,none,2019-03-27 10:34:53.041496,function,org.apache.syncope.client.ui.commons.ConnIdSpe...,git@github.com:apache/syncope.git,114c412afbfba24ffb4fbc804e5308a823a16a78,/client/idrepo/ui/src/main/java/org/apache/syn...,35,37,https://github.com/apache/syncope/blob/114c412...,1
1,527,6,5771277,long method,none,2019-03-27 10:34:53.042443,function,org.apache.syncope.client.ui.commons.ConnIdSpe...,git@github.com:apache/syncope.git,114c412afbfba24ffb4fbc804e5308a823a16a78,/client/idrepo/ui/src/main/java/org/apache/syn...,35,37,https://github.com/apache/syncope/blob/114c412...,1
2,528,6,5786929,blob,critical,2019-03-27 10:37:38.107923,class,org.apache.tez.runtime.library.common.writers....,git@github.com:apache/tez.git,d5675c332497c1ac1dedefdf91e87476b5c0d7a9,/tez-runtime-library/src/main/java/org/apache/...,89,1427,https://github.com/apache/tez/blob/d5675c33249...,1
3,529,6,5786929,data class,critical,2019-03-27 10:37:38.109068,class,org.apache.tez.runtime.library.common.writers....,git@github.com:apache/tez.git,d5675c332497c1ac1dedefdf91e87476b5c0d7a9,/tez-runtime-library/src/main/java/org/apache/...,89,1427,https://github.com/apache/tez/blob/d5675c33249...,1
4,530,6,5788107,feature envy,none,2019-03-27 10:37:49.627100,function,org.apache.tika.parser.ocr.TesseractOCRConfig#...,git@github.com:apache/tika.git,4131c6e30f2e0eb1feb85e0f7576531d4e830468,/tika-parsers/src/main/java/org/apache/tika/pa...,531,534,https://github.com/apache/tika/blob/4131c6e30f...,1


*Pre-processing of dataset*

In [30]:
# Filter by relevant projects and copy dataset to perfom changes
original = original[original['is_from_industry_relevant_project'] == '1'].copy()
# Convert lines to numeric where suitable
original['start_line'] = pd.to_numeric(original['start_line'], errors='coerce')
original['end_line'] = pd.to_numeric(original['end_line'], errors='coerce')
original = original.dropna(subset=['start_line', 'end_line'])
original['start_line'] = original['start_line'].astype(int)
original['end_line'] = original['end_line'].astype(int)
# remove lines with empty core data 
original = original.dropna(subset=['repository', 'commit_hash', 'path'])

In [31]:
original.shape

(12710, 15)

## 2. Busca do código nos repositórios

O dataset **não traz o code snippet**, apenas as indicações (repositório, commit, arquivo, `start_line`, `end_line`). Por isso foi necessário buscar o conteúdo nos repositórios. O script `collect_mlcq_samples_with_source_code.py` faz essa coleta via API do GitHub e gera um CSV com os campos originais mais `code_snippet` e `file_content`.

In [ ]:
# Dataset já salvo; descomente abaixo para refazer a busca do source code nos repositórios
# original.to_csv(BASE / "mlcq_filtered_dataset.csv", sep=";", index=False)
#
# import sys
# sys.path.insert(0, str(BASE))
# from collect_mlcq_samples_with_source_code import process_dataset
#
# process_dataset(
#     original_csv_file=str(BASE / "mlcq_filtered_dataset.csv"),
#     json_file=str(BASE / "original_mlcq_samples.json"),
#     csv_file_with_source_code=str(BASE / "mlcq_enriched_dataset.csv"),
# )

Fetching code + source files: 25it [00:06,  4.15it/s]

Failed to fetch file: https://raw.githubusercontent.com/apache/uima-ruta/08efb111a775b4ba01cf048ac699aa9c22188325/ruta-core/src/main/java/org/apache/uima/ruta/action/TrimAction.java (status: 404)
Failed to fetch file: https://raw.githubusercontent.com/apache/uima-ruta/08efb111a775b4ba01cf048ac699aa9c22188325/ruta-core/src/main/java/org/apache/uima/ruta/action/TrimAction.java (status: 404)


Fetching code + source files: 36it [00:08,  4.63it/s]

Failed to fetch file: https://raw.githubusercontent.com/apache/uima-ruta/08efb111a775b4ba01cf048ac699aa9c22188325/ruta-ep-addons/src/main/java/org/apache/uima/ruta/cde/ui/DocumentTableLabelProvider.java (status: 404)
Failed to fetch file: https://raw.githubusercontent.com/apache/uima-ruta/08efb111a775b4ba01cf048ac699aa9c22188325/ruta-ep-addons/src/main/java/org/apache/uima/ruta/cde/ui/DocumentTableLabelProvider.java (status: 404)


Fetching code + source files: 50it [00:12,  4.67it/s]

...

Saved batch of 46 entries.


Fetching code + source files: 12701it [3:43:29,  5.24s/it]

Saved batch of 50 entries.


Fetching code + source files: 12710it [3:43:31,  1.06s/it]


Saved batch of 10 entries.
Completed JSON saving. Output: /home/marcelo/Doutorado/llm-code-review-on-smells/datasets/original_mlcq_samples.json
Saving CSV to: /home/marcelo/Doutorado/llm-code-review-on-smells/datasets/mlcq_samples_with_source_code.csv
CSV saved successfully.

=== Processing Summary ===
Total rows processed: 12710
Successfully processed: 11603
Skipped (fetch failures): 1107
Errors (parsing issues): 0
Skipped entries saved to: /home/marcelo/Doutorado/llm-code-review-on-smells/datasets/mlcq_samples_with_source_code_skipped.csv


## 3. CSV com source code

Após a coleta, o CSV gerado contém os snippets e o conteúdo completo do arquivo.

In [ ]:
with_source = pd.read_csv(BASE / "mlcq_enriched_dataset.csv")
with_source.head()

,id,repo_url,commit_hash,file_path,start_line,end_line,code_snippet,file_content,smell,severity,type,code_name
0,526,git@github.com:apache/syncope.git,114c412afbfba24ffb4fbc804e5308a823a16a78,/client/idrepo/ui/src/main/java/org/apache/syn...,35,37,private ConnIdSpecialName() {\n // ...,/*\n * Licensed to the Apache Software Foundat...,feature envy,none,function,org.apache.syncope.client.ui.commons.ConnIdSpe...
1,527,git@github.com:apache/syncope.git,114c412afbfba24ffb4fbc804e5308a823a16a78,/client/idrepo/ui/src/main/java/org/apache/syn...,35,37,private ConnIdSpecialName() {\n // ...,/*\n * Licensed to the Apache Software Foundat...,long method,none,function,org.apache.syncope.client.ui.commons.ConnIdSpe...
2,528,git@github.com:apache/tez.git,d5675c332497c1ac1dedefdf91e87476b5c0d7a9,/tez-runtime-library/src/main/java/org/apache/...,89,1427,public class UnorderedPartitionedKVWriter exte...,/**\n * Licensed to the Apache Software Founda...,blob,critical,class,org.apache.tez.runtime.library.common.writers....
3,529,git@github.com:apache/tez.git,d5675c332497c1ac1dedefdf91e87476b5c0d7a9,/tez-runtime-library/src/main/java/org/apache/...,89,1427,public class UnorderedPartitionedKVWriter exte...,/**\n * Licensed to the Apache Software Founda...,data class,critical,class,org.apache.tez.runtime.library.common.writers....
4,530,git@github.com:apache/tika.git,4131c6e30f2e0eb1feb85e0f7576531d4e830468,/tika-parsers/src/main/java/org/apache/tika/pa...,531,534,public String getImageMagickPath() {\n\n ...,/*\n * Licensed to the Apache Software Foundat...,feature envy,none,function,org.apache.tika.parser.ocr.TesseractOCRConfig#...


In [35]:
with_source.shape

(11603, 12)

## 4. Gerar SQL para a base de dados

Geração do arquivo de inserts a partir do CSV com source para inserção na base.

In [ ]:
# Inserts SQL já gerados; descomente abaixo para refazer
# import sys
# sys.path.insert(0, str(BASE / "database-storage"))
# from generate_sql_inserts_from_csv import generate_sql_inserts
#
# generate_sql_inserts(
#     input_csv=str(BASE / "mlcq_enriched_dataset.csv"),
#     output_sql=str(BASE / "mlcq_inserts.sql"),
# )